# সহজ ভাষায় Notebook Guide

এই notebook-এ theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে—যাতে code-এর language-এর সাথে পরিচিত থেকেও concept সহজে বোঝা যায়।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. Run করার আগে expected output কী হতে পারে লিখে ভাবুন।
3. Output-এর metric, shape এবং visualization explanation-এর সাথে compare করুন।
4. Error হলে import, file path, data shape এবং dependency একে একে check করুন।
5. Notebook শেষে নিজের ভাষায় লিখুন: এটি কোন problem solve করেছে, কীভাবে করেছে এবং limitation কী।

> **Important:** Notebook-এর সব cell successful run হলেই result correct প্রমাণ হয় না; data leakage, wrong assumption এবং misleading metric-ও validate করতে হবে।

# Document Indexer & Semantic Search Demo: ChromaDB + Chunking Strategies

> **Focus Area:** ভেক্টর ডেটাবেস (Vector Databases - Retrieval Foundations for RAG)
> **Topics:** ChromaDB বেসিকস (PersistentClient, Collection, Embedding); সিমেন্টিক বনাম কিওয়ার্ড সার্চ; টেক্সট চাংকিং স্ট্র্যাটেজি (Fixed-Size, Recursive, Overlap)
> **Achievement:** একটা এমপ্লয়ি হ্যান্ডবুক-স্টাইল ডকুমেন্টকে বিভিন্ন চাংকিং স্ট্র্যাটেজি দিয়ে ভাগ করা, ChromaDB-তে ইনডেক্স করা, এবং সিমেন্টিক ও কিওয়ার্ড সার্চের মধ্যে পার্থক্য হাতে-কলমে দেখা — যা পরের ক্লাসের RAG বট (Project 10)-এর ভিত্তি।

---

## 1. Topic: ChromaDB, Semantic Search, and Text Chunking

Week 10-এ আমরা `sentence-transformers` দিয়ে টেক্সটকে এমবেডিং ভেক্টরে রূপান্তর করেছিলাম, আর NumPy দিয়ে ম্যানুয়ালি cosine similarity হিসাব করেছিলাম। কিন্তু বাস্তবে হাজার হাজার ডকুমেন্ট চাংক থাকলে প্রতিবার সবগুলোর সাথে one-by-one তুলনা করা স্লো হয়ে যায়। **ChromaDB** এই সমস্যার সমাধান — একটা লোকাল, ওপেন-সোর্স ভেক্টর ডেটাবেস, যা কোনো সার্ভার সেটআপ ছাড়াই সরাসরি পাইথন থেকে চালানো যায় এবং দ্রুততম সময়ে সবচেয়ে কাছাকাছি ভেক্টর খুঁজে বের করে।

এই নোটবুকে আমরা তিনটা জিনিস হাতে-কলমে দেখব:

* **ChromaDB Basics** — `PersistentClient`, `collection`, ম্যানুয়াল embedding দিয়ে চাংক যোগ করা এবং `collection.query()` চালানো।
* **Chunking Strategies** — Fixed-Size বনাম Recursive/Paragraph-Aware চাংকিং, এবং Overlap কেন দরকার।
* **Semantic vs. Keyword Search** — একই কোয়েরি দুই পদ্ধতিতে চালিয়ে সরাসরি পার্থক্য দেখা।

**Project Goal (এই ক্লাসের ডেমো, পোর্টফোলিও প্রজেক্ট না):** একটা সিন্থেটিক এমপ্লয়ি হ্যান্ডবুক ইনডেক্স করে ChromaDB-তে সেভ করা, তারপর সেখান থেকে সিমেন্টিক সার্চ চালিয়ে দেখানো।

---

## 2. Why It Is Related

পরের ক্লাসেই আমরা একটা RAG (Retrieval-Augmented Generation) বট বানাব (Project 10), যেটা আপলোড করা PDF থেকে প্রশ্নের উত্তর দেবে। RAG বট আসলে দুটো আলাদা স্কিলের সমন্বয়: (১) সঠিক তথ্য খুঁজে বের করা (Retrieval), আর (২) সেই তথ্য দিয়ে উত্তর জেনারেট করা (Generation)। এই নোটবুকে আমরা প্রথম অংশটা — Retrieval — গভীরভাবে অনুশীলন করছি, যাতে পরের ক্লাসে সেটার ওপর LLM জুড়ে দেওয়াটা সহজ হয়। একটা দুর্বল Retrieval System-এর ওপর RAG বানালে বট ভুল তথ্য দিয়ে answer generate করবে (hallucinate করবে) — তাই এই ফাউন্ডেশনটা RAG-এর সবচেয়ে গুরুত্বপূর্ণ অংশ।

---

## 3. How It Works

### 3.1 ChromaDB: PersistentClient, Collection, Embedding

`chromadb.PersistentClient(path=...)` দিয়ে একটা লোকাল, ডিস্কে-পার্সিস্টেন্ট ক্লায়েন্ট তৈরি হয় — কোনো সার্ভার বা নেটওয়ার্ক কল লাগে না। এর ভেতরে `get_or_create_collection()` দিয়ে একটা **collection** (একটা SQL টেবিলের মতো, কিন্তু ভেক্টরের জন্য) বানানো হয়। প্রতিটা চাংক `collection.add()`-এ যোগ হয় চারটা জিনিসসহ: `ids` (ইউনিক আইডি), `embeddings` (আমরা নিজেরাই `sentence-transformers` দিয়ে বানাবো), `documents` (আসল টেক্সট), আর `metadatas` (source ফাইল, page নম্বর — পরে citation আর filtering-এর ভিত্তি)। এই নোটবুকে আমরা Week 10-এর সাথে সামঞ্জস্য রাখতে একই `paraphrase-MiniLM-L3-v2` মডেল ব্যবহার করব (দেখুন `docs/adr/0004-week10-class19-embedding-model.md`)।

### 3.2 Text Chunking Strategies

একটা পুরো ডকুমেন্ট একবারে এমবেড করা যায় না — এমবেডিং মডেলের একটা ম্যাক্স ইনপুট সাইজ থাকে, আর ছোট চাংক বেশি নির্দিষ্ট (precise) রিট্রিভাল দেয়।

* **Fixed-Size Chunking:** প্রতি N ক্যারেক্টারে নির্বিশেষে ভাগ করা, বাক্য/প্যারাগ্রাফ কোথায় শেষ হচ্ছে তা না দেখেই। সহজ, কিন্তু মাঝ-বাক্যে কেটে ফেলতে পারে — অর্থ ভেঙে যায়।
* **Recursive/Paragraph-Aware Chunking:** আগে প্যারাগ্রাফ ব্রেক (`\n\n`) দিয়ে ভাগ করার চেষ্টা করা; কোনো প্যারাগ্রাফ তখনও টার্গেট সাইজের চেয়ে বড় থাকলে সেটাকে বাক্যে (sentence) ভেঙে আবার জোড়া লাগানো, যতক্ষণ না চাংক টার্গেট সাইজের নিচে আসে। মাঝ-বাক্যে কাটা এড়ানো যায়।
* **Overlap:** প্রতিটা চাংকের শেষের কিছু অংশ পরের চাংকের শুরুতেও রাখা, যাতে কোনো তথ্য ঠিক চাংকের বর্ডারে পড়ে গিয়ে হারিয়ে না যায়।

চাংক সাইজ একটা ট্রেড-অফ (context.md সেকশন 4.2 দেখুন): ছোট চাংক = বেশি নির্দিষ্ট রিট্রিভাল কিন্তু কম কনটেক্সট; বড় চাংক = বেশি কনটেক্সট কিন্তু একাধিক বিষয় মিশে গিয়ে এমবেডিং "গড়পড়তা" অর্থ ধরে ফেলে (ডাইলিউশন)। ব্যবহারিক অভিজ্ঞতা থেকে ৩০০-৮০০ ক্যারেক্টার রেঞ্জে ভালো ব্যালেন্স পাওয়া যায়, সাথে ~১০-১৫% ওভারল্যাপ।

### 3.3 Semantic vs. Keyword Search

**Keyword Search** (traditional, যেমন Python `in` বা SQL `LIKE`): হুবহু শব্দ থাকা চাংকগুলোই পাওয়া যায় — সমার্থক শব্দ (synonym) মিস হয়ে যায়। **Semantic Search** (embedding-ভিত্তিক): "sick leave" আর "medical absence" — দুটোই কাছাকাছি ভেক্টরে থাকে, তাই semantic search উভয় ধরনের চাংক খুঁজে পায়, হুবহু শব্দ না মিললেও। নিচের সেকশন 6-এ আমরা এই পার্থক্যটা সরাসরি একটা রিয়েল উদাহরণ দিয়ে দেখাব।

---

## 4. Achievement: ChromaDB Basics — Client, Collection, Add, Query

প্রথমে একটা `PersistentClient` বানিয়ে একটা collection তৈরি করব, তারপর কয়েকটা উদাহরণ টেক্সট চাংক তাদের `sentence-transformers` এমবেডিং আর metadata (source, page) সহ যোগ করব। শেষে `collection.query()` দিয়ে একটা সিমেন্টিক কোয়েরি চালাব।

In [1]:
# pip install chromadb sentence-transformers  (প্রথমবার চালানোর আগে দরকার হলে uncomment করুন)
# !pip install chromadb sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer

# Week 10-এর সাথে সামঞ্জস্য রাখতে একই লোকাল মডেল (docs/adr/0004) -- ছোট (~61MB), CPU-তেই ফাস্ট
embed_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

# লোকাল, ডিস্কে-পার্সিস্টেন্ট ক্লায়েন্ট -- কোনো সার্ভার বা নেটওয়ার্ক কল লাগে না
client = chromadb.PersistentClient(path="./chroma_db_demo")
collection = client.get_or_create_collection(name="handbook_demo")

print(f"Collection name: {collection.name}")
print(f"Existing item count: {collection.count()}")


C:\Users\81250\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 18642.86it/s]

Collection name: handbook_demo
Existing item count: 4


In [2]:
# --- কয়েকটা উদাহরণ চাংক, তাদের সোর্স ও পেজ নম্বর মেটাডেটাসহ ---
example_chunks = [
    {"id": "demo_1", "text": "Employees must submit leave requests at least 3 business days in advance through the HR portal.", "source": "handbook.pdf", "page": 12},
    {"id": "demo_2", "text": "All full-time staff are entitled to 14 days of paid sick leave per calendar year.", "source": "handbook.pdf", "page": 14},
    {"id": "demo_3", "text": "The office provides a hybrid work policy allowing employees to work remotely up to two days per week.", "source": "handbook.pdf", "page": 27},
    {"id": "demo_4", "text": "Expense reports must be filed within 30 days of the expense date to be eligible for reimbursement.", "source": "handbook.pdf", "page": 41},
]

# ম্যানুয়ালি এমবেডিং বানানো -- encode() একটা numpy array রিটার্ন করে, ChromaDB-র জন্য list-এ কনভার্ট করা লাগবে
embeddings = embed_model.encode([c["text"] for c in example_chunks]).tolist()

collection.upsert(
    ids=[c["id"] for c in example_chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in example_chunks],
    metadatas=[{"source": c["source"], "page": c["page"]} for c in example_chunks],
)

print(f"Collection item count after add: {collection.count()}")


Collection item count after add: 4


In [3]:
# --- সিমেন্টিক কোয়েরি: query_embeddings দিয়ে সবচেয়ে কাছের চাংক খোঁজা ---
query_text = "How many paid sick days do I get?"
query_embedding = embed_model.encode([query_text]).tolist()

results = collection.query(query_embeddings=query_embedding, n_results=2)

print(f"Query: {query_text}\n")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"  distance={dist:.4f}  source={meta['source']} (page {meta['page']})")
    print(f"  text: {doc}\n")


Query: How many paid sick days do I get?

  distance=12.5198  source=handbook.pdf (page 14)
  text: All full-time staff are entitled to 14 days of paid sick leave per calendar year.

  distance=27.2534  source=handbook.pdf (page 27)
  text: The office provides a hybrid work policy allowing employees to work remotely up to two days per week.



**খেয়াল করুন:** কোয়েরিতে "sick days" শব্দটা থাকলেও চাংক টেক্সটে "sick leave" লেখা — তবুও এমবেডিং দুটো কাছাকাছি ভেক্টরে বসেছে বলে ChromaDB সঠিক চাংকটাই সবচেয়ে কম distance-এ ফেরত দিয়েছে (distance যত কম, similarity তত বেশি)। এটাই semantic search-এর মূল শক্তি, যা সেকশন 6-এ আরও স্পষ্টভাবে keyword search-এর সাথে তুলনা করে দেখানো হবে।

---

## 5. Achievement: Chunking Strategy Comparison — Fixed-Size vs. Recursive

এবার একটা মাল্টি-প্যারাগ্রাফ সিন্থেটিক "employee handbook" টেক্সট নিয়ে দেখব Fixed-Size চাংকিং কীভাবে মাঝ-বাক্যে কেটে ফেলে, আর Recursive/Paragraph-Aware চাংকিং কীভাবে প্যারাগ্রাফ বর্ডার সম্মান করে।

In [4]:
handbook_text = """Leave Policy: Employees must submit leave requests at least three business days in advance through the HR portal. Approved leave will be reflected in the payroll system within one business day. Unused annual leave up to five days may be carried forward into the next calendar year, but any balance beyond that will be forfeited on December 31st.

Sick Leave: All full-time staff are entitled to fourteen days of paid sick leave per calendar year. A medical certificate is required for any sick leave exceeding two consecutive days. Employees who are unable to work due to illness for more than five consecutive days should notify their manager and HR jointly so that workload coverage can be arranged in advance.

Remote Work: The company offers a hybrid work policy allowing employees to work remotely up to two days per week, subject to manager approval. Employees must remain reachable during core hours (10 AM to 4 PM local time) regardless of their work location. Any equipment issued for remote work remains company property and must be returned upon termination of employment.

Expense Reimbursement: Expense reports must be filed within thirty days of the expense date to be eligible for reimbursement. Reports missing original receipts will be rejected unless a manager provides written approval. Reimbursements are typically processed within two payroll cycles of report submission.

Code of Conduct: Employees are expected to treat colleagues, clients, and vendors with professionalism and respect at all times. Violations of the code of conduct may result in disciplinary action up to and including termination. Any concerns should be reported confidentially to HR through the designated ethics hotline."""

print(f"Total characters: {len(handbook_text)}")
print(f"Total paragraphs: {len(handbook_text.split(chr(10)+chr(10)))}")


Total characters: 1715
Total paragraphs: 5


In [5]:
# --- (a) Fixed-Size Chunking: বাক্য/প্যারাগ্রাফ বর্ডার উপেক্ষা করে সোজা N ক্যারেক্টারে কাটা ---
def fixed_size_chunks(text, chunk_size=200):
    # নির্বিশেষে প্রতি chunk_size ক্যারেক্টারে কাটা হচ্ছে -- মাঝ-বাক্যেও কাটতে পারে
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

fixed_chunks = fixed_size_chunks(handbook_text, chunk_size=200)

print(f"Fixed-size chunk count: {len(fixed_chunks)}\n")
for i, chunk in enumerate(fixed_chunks[:4]):
    print(f"--- Fixed Chunk {i} ({len(chunk)} chars) ---")
    print(repr(chunk))
    print()


Fixed-size chunk count: 9

--- Fixed Chunk 0 (200 chars) ---
'Leave Policy: Employees must submit leave requests at least three business days in advance through the HR portal. Approved leave will be reflected in the payroll system within one business day. Unused'

--- Fixed Chunk 1 (200 chars) ---
' annual leave up to five days may be carried forward into the next calendar year, but any balance beyond that will be forfeited on December 31st.\n\nSick Leave: All full-time staff are entitled to fourt'

--- Fixed Chunk 2 (200 chars) ---
'een days of paid sick leave per calendar year. A medical certificate is required for any sick leave exceeding two consecutive days. Employees who are unable to work due to illness for more than five c'

--- Fixed Chunk 3 (200 chars) ---
'onsecutive days should notify their manager and HR jointly so that workload coverage can be arranged in advance.\n\nRemote Work: The company offers a hybrid work policy allowing employees to work remote'



**খেয়াল করুন:** উপরের চাংকগুলোর মধ্যে অনেকগুলোই একটা বাক্যের মাঝখানে শুরু বা শেষ হচ্ছে। এমবেডিং মডেল এই অসম্পূর্ণ বাক্য থেকে সঠিক অর্থ বুঝতে সমস্যায় পড়ে।

In [6]:
import re

# --- (b) Recursive/Paragraph-Aware Chunking ---
def split_into_sentences(paragraph):
    # সাধারণ সেন্টেন্স-বাউন্ডারি স্প্লিট: '.', '!', '?' এর পর স্পেস দেখলে ভাঙা
    sentences = re.split(r'(?<=[.!?])\s+', paragraph.strip())
    return [s for s in sentences if s]

def recursive_chunks(text, target_size=300):
    # ধাপ ১: আগে প্যারাগ্রাফ ব্রেক (\n\n) দিয়ে ভাগ করার চেষ্টা
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    for para in paragraphs:
        if len(para) <= target_size:
            # প্যারাগ্রাফ নিজেই টার্গেট সাইজের নিচে -- সরাসরি একটা চাংক
            chunks.append(para)
        else:
            # ধাপ ২: প্যারাগ্রাফ এখনো বড় -- বাক্যে ভেঙে আবার জোড়া লাগানো
            sentences = split_into_sentences(para)
            current = ""
            for sent in sentences:
                if current and len(current) + len(sent) + 1 > target_size:
                    chunks.append(current.strip())
                    current = sent
                else:
                    current = (current + " " + sent).strip()
            if current:
                chunks.append(current.strip())
    return chunks

recursive_chunk_list = recursive_chunks(handbook_text, target_size=300)

print(f"Recursive chunk count: {len(recursive_chunk_list)}\n")
for i, chunk in enumerate(recursive_chunk_list):
    print(f"--- Recursive Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()


Recursive chunk count: 10

--- Recursive Chunk 0 (193 chars) ---
Leave Policy: Employees must submit leave requests at least three business days in advance through the HR portal. Approved leave will be reflected in the payroll system within one business day.

--- Recursive Chunk 1 (151 chars) ---
Unused annual leave up to five days may be carried forward into the next calendar year, but any balance beyond that will be forfeited on December 31st.

--- Recursive Chunk 2 (184 chars) ---
Sick Leave: All full-time staff are entitled to fourteen days of paid sick leave per calendar year. A medical certificate is required for any sick leave exceeding two consecutive days.

--- Recursive Chunk 3 (180 chars) ---
Employees who are unable to work due to illness for more than five consecutive days should notify their manager and HR jointly so that workload coverage can be arranged in advance.

--- Recursive Chunk 4 (254 chars) ---
Remote Work: The company offers a hybrid work policy allowing emplo

**খেয়াল করুন:** Recursive চাংকিং-এ প্রতিটা চাংক সবসময় সম্পূর্ণ বাক্যে শেষ হচ্ছে — কোনো বাক্য মাঝপথে কাটা যাচ্ছে না, এবং প্যারাগ্রাফ বর্ডার যতটা সম্ভব সম্মান করা হচ্ছে। এই কারণেই প্রোডাকশন RAG সিস্টেমে (যেমন LangChain-এর `RecursiveCharacterTextSplitter`) এই স্ট্র্যাটেজিই ডিফল্ট হিসেবে ব্যবহৃত হয়।

In [7]:
# --- Overlap উদাহরণ: চাংকের বর্ডারে তথ্য হারিয়ে যাওয়া ঠেকানো ---
def chunks_with_overlap(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        # পরের চাংক শুরু হবে (chunk_size - overlap) দূরত্বে, ফলে শেষের overlap অংশ পুনরাবৃত্তি হয়
        start += chunk_size - overlap
    return chunks

overlap_chunks = chunks_with_overlap(handbook_text, chunk_size=200, overlap=50)

print(f"Overlap chunk count: {len(overlap_chunks)}\n")
print("--- Chunk 0 tail (last 60 chars) ---")
print(repr(overlap_chunks[0][-60:]))
print("\n--- Chunk 1 head (first 60 chars) ---")
print(repr(overlap_chunks[1][:60]))
print("\nOverlap অংশটা দুই চাংকেই দেখা যাচ্ছে -- কোনো একটা গুরুত্বপূর্ণ বাক্য ঠিক বর্ডারে পড়ে গেলেও সেটা অন্তত একটা চাংকে সম্পূর্ণ থাকবে।")


Overlap chunk count: 12

--- Chunk 0 tail (last 60 chars) ---
'lected in the payroll system within one business day. Unused'

--- Chunk 1 head (first 60 chars) ---
'the payroll system within one business day. Unused annual le'

Overlap অংশটা দুই চাংকেই দেখা যাচ্ছে -- কোনো একটা গুরুত্বপূর্ণ বাক্য ঠিক বর্ডারে পড়ে গেলেও সেটা অন্তত একটা চাংকে সম্পূর্ণ থাকবে।


---

## 6. Achievement: Semantic Search vs. Keyword Search — Side by Side

এবার recursive চাংকগুলো ChromaDB-তে ইনডেক্স করে একটা কোয়েরি চালাব — কোয়েরিতে "medical absence" শব্দ ব্যবহার করা হবে, যদিও হ্যান্ডবুক টেক্সটে সবচেয়ে প্রাসঙ্গিক চাংকটা মূলত "sick leave" শব্দ দিয়ে লেখা। আমরা দেখব semantic search কীভাবে অর্থগতভাবে কাছাকাছি চাংক খুঁজে পায়, আর naive keyword matching শুধু হুবহু substring থাকা চাংকই ধরতে পারে (এবং সেটা না থাকলে কিছুই ফেরত দিতে পারে না)।

In [8]:
# --- হ্যান্ডবুক চাংকগুলো নতুন একটা collection-এ ইনডেক্স করা ---
handbook_collection = client.get_or_create_collection(name="handbook_search_demo")

chunk_ids = [f"chunk_{i}" for i in range(len(recursive_chunk_list))]
chunk_embeddings = embed_model.encode(recursive_chunk_list).tolist()

handbook_collection.upsert(
    ids=chunk_ids,
    embeddings=chunk_embeddings,
    documents=recursive_chunk_list,
    metadatas=[{"source": "employee_handbook.txt", "chunk_index": i} for i in range(len(recursive_chunk_list))],
)

print(f"Indexed {handbook_collection.count()} chunks.")


Indexed 10 chunks.

In [9]:
# --- Query যেখানে সরাসরি "medical absence" phrase হুবহু নেই, শুধু সমার্থক "sick leave" আছে ---
query = "medical absence policy"

# (a) Semantic Search: embedding cosine similarity দিয়ে
query_vec = embed_model.encode([query]).tolist()
semantic_results = handbook_collection.query(query_embeddings=query_vec, n_results=2)

print(f"Query: '{query}'\n")
print("=== Semantic Search Results (embedding-based) ===")
for doc, dist in zip(semantic_results["documents"][0], semantic_results["distances"][0]):
    print(f"  distance={dist:.4f}")
    print(f"  {doc}\n")

# (b) Keyword Search: naive Python `in` substring check -- কোনো মডেল/এমবেডিং লাগে না
print("=== Keyword Search Results (naive substring match on 'medical absence') ===")
keyword_matches = [c for c in recursive_chunk_list if "medical absence" in c.lower()]
if keyword_matches:
    for m in keyword_matches:
        print(f"  {m}\n")
else:
    print("  (কোনো চাংকে হুবহু 'medical absence' substring নেই -- keyword search কিছুই খুঁজে পেল না)")


Query: 'medical absence policy'

=== Semantic Search Results (embedding-based) ===
  distance=33.0388
  Sick Leave: All full-time staff are entitled to fourteen days of paid sick leave per calendar year. A medical certificate is required for any sick leave exceeding two consecutive days.

  distance=35.3495
  Employees who are unable to work due to illness for more than five consecutive days should notify their manager and HR jointly so that workload coverage can be arranged in advance.

=== Keyword Search Results (naive substring match on 'medical absence') ===
  (কোনো চাংকে হুবহু 'medical absence' substring নেই -- keyword search কিছুই খুঁজে পেল না)


**খেয়াল করুন:** Semantic search কোয়েরির সবচেয়ে কাছের চাংকটা ঠিকই খুঁজে বের করেছে — যেটাতে "sick leave" শব্দ আছে, "medical absence" ফ্রেজ হুবহু না থাকলেও (embedding দুটো phrase-কেই কাছাকাছি ভেক্টরে রাখে, কারণ অর্থ একই)। কিন্তু naive keyword matching শুধু হুবহু "medical absence" স্ট্রিং আছে এমন চাংক খুঁজেছে — যা হ্যান্ডবুকে নাও থাকতে পারে, ফলে কোনো ফলাফল না-ও পাওয়া যেতে পারে। এটাই context.md সেকশন 3.3-এ বলা মূল পার্থক্য — সমার্থক শব্দ (synonym) থাকলে keyword search miss করে, semantic search করে না। তবে মনে রাখবেন — নির্দিষ্ট পলিসি নম্বর বা কোড (যেমন "Policy #4471") খোঁজার সময় keyword search-ই বরং বেশি নির্ভরযোগ্য, কারণ embedding মডেল নির্দিষ্ট আইডেন্টিফায়ারের "অর্থ" বোঝে না।

---

## 7. Summary

আজকে আমরা Retrieval-এর তিনটা বিল্ডিং ব্লক হাতে-কলমে দেখলাম:

1. **ChromaDB Basics** — `PersistentClient` দিয়ে একটা লোকাল, সার্ভার-ছাড়া ভেক্টর ডেটাবেস তৈরি করা, চাংক তাদের embedding, document text আর metadata (source, page) সহ যোগ করা, এবং `collection.query()` দিয়ে সিমেন্টিক কোয়েরি চালানো।
2. **Chunking Strategies** — Fixed-Size চাংকিং কীভাবে মাঝ-বাক্যে কেটে অর্থ ভেঙে ফেলে, আর Recursive/Paragraph-Aware চাংকিং কীভাবে প্যারাগ্রাফ আর বাক্য বর্ডার সম্মান করে সম্পূর্ণ অর্থবহ চাংক তৈরি করে; Overlap কীভাবে বর্ডারে তথ্য হারিয়ে যাওয়া ঠেকায়।
3. **Semantic vs. Keyword Search** — একই কোয়েরি দুই পদ্ধতিতে চালিয়ে সরাসরি দেখলাম semantic search সমার্থক শব্দ থাকা চাংকও খুঁজে পায়, keyword search শুধু হুবহু স্ট্রিং মিললেই পায়।

এই তিনটা মিলেই তৈরি হয় একটা Retrieval System-এর ভিত্তি — যেটার ওপর পরের ক্লাসে (Class 24) LLM জুড়ে দিয়ে একটা সম্পূর্ণ RAG বট (Project 10) বানানো হবে।

---

## 🧠 Brain Teasers & Exercises (নিজে চেষ্টা করুন)

1. **Chunk Size Experiment**: উপরের `recursive_chunks()` ফাংশন `target_size=200`, `500`, আর `1500` দিয়ে আবার চালান, তারপর সেকশন 6-এর একই কোয়েরি (`"medical absence policy"`) `handbook_collection.query()`-তে চালিয়ে দেখুন কোন সাইজে সবচেয়ে প্রাসঙ্গিক (কম distance-এর) চাংক পাওয়া যায়।
2. **Semantic Search Failure Case**: এমন একটা কোয়েরি ভাবুন যেখানে semantic search ব্যর্থ হবে কিন্তু keyword search সফল হবে — যেমন `"Section 4.2"` বা `"Policy #4471"`-এর মতো একটা নির্দিষ্ট কোড `recursive_chunk_list`-এ যোগ করে দুই পদ্ধতিতেই খুঁজে দেখুন কোনটা সফল হয়, আর কেন।
3. **Overlap Necessity**: `chunks_with_overlap()`-এ `overlap=0` দিয়ে চালান এবং `overlap=50`-এর সাথে তুলনা করুন। এমন একটা গুরুত্বপূর্ণ বাক্য খুঁজে বের করুন যেটা `overlap=0`-এ ঠিক দুই চাংকের বর্ডারে ভেঙে গেছে, কিন্তু `overlap=50`-এ অন্তত একটা চাংকে সম্পূর্ণ আছে।